In [ ]:
!pip install python-docx
!pip install -U transformers
!pip install -q trl bert-score openai
!pip uninstall -y trl
!pip install trl==0.11.3
!pip install -q accelerate>=1.8.0
!pip install -q bitsandbytes>=0.46.1
!pip install -q datasets

import time
import random
import re
import pandas as pd
from docx import Document
import numpy as np
from datasets import Dataset
import os

import torch
torch.cuda.empty_cache()

import torch
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
from bert_score import BERTScorer
import openai
from datasets import Dataset
import numpy as np
import gc
from tqdm import tqdm

import bert_score
import transformers
import json
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

torch.cuda.empty_cache()
import gc
gc.collect()

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 113.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.2 MB/s eta 0:00:00
Found existing installation: trl 1.4.0
Uninstalling trl-1.4.0:
  Successfully uninstalled trl-1.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.0 MB/s eta 0:00:00
Mounted at /content/drive


Загружаем модель и токенизатор

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

Подключаем модель с openrouter по api

In [ ]:
USE_LLM_REWARD = True
OPENROUTER_API_KEY = ""
JUDGE_MODEL = "openai/gpt-oss-120b:free"

bertscorer = BERTScorer(lang="ru", rescale_with_baseline=False, device='cuda')

if USE_LLM_REWARD:
    openrouter_client = openai.OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY
    )
last_llm_score = 0.0

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ставим random_seed, чтобы ответ воспроизводился

In [ ]:
def set_random_seed(seed: int = 27):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True

set_random_seed()

Функции для LLM и reward

In [ ]:
llm_cache = {}

def get_llm_score(generated_text, source_text, topic):
    cache_key = (generated_text, source_text, topic)
    if cache_key in llm_cache:
        return llm_cache[cache_key]

    if not USE_LLM_REWARD:
        return None

    model = JUDGE_MODEL

    prompt = f"""Ты - социолог-эксперт в открытом кодировании интервью.
Тема интервью: {topic}
Текст интервью: {source_text}

Сгенерированные коды и цитаты (в формате <код>цитата</код>):
{generated_text}

Оцени результат по трём критериям (каждый от 0 до 1):
1. Релевантность - насколько код соответствует содержанию фрагмента интервью с учётом темы.
2. Когерентность - насколько формулировка кода грамматически корректна и стилистически приемлема.
3. Теоретический инсайт - степень соответствия кода концептуальным ожиданиям.

Ответ дай строго в формате: число, число, число (например: 0.85, 0.90, 0.75). Не пиши никаких пояснений."""

    max_retries = 5
    base_delay = 1.0
    max_delay = 30.0

    for attempt in range(max_retries):
        try:
            response = openrouter_client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=50
            )

            if response and hasattr(response, 'choices') and response.choices:
                content = response.choices[0].message.content
                if content:

                    numbers = re.findall(r"(\d+\.?\d*)", content)
                    if len(numbers) >= 3:
                        try:
                            scores = [float(n) for n in numbers[:3]]
                            mean_score = np.mean(scores)
                            mean_score = max(0.0, min(1.0, mean_score))
                            llm_cache[cache_key] = mean_score
                            return mean_score
                        except ValueError:
                            print(f"LLM warning: non-numeric numbers: {numbers}", flush=True)
                    else:
                        print(f"LLM warning: expected 3 numbers, got {len(numbers)}. Content: {content[:200]}", flush=True)
                else:
                    print("LLM warning: empty content", flush=True)
            else:
                print("LLM warning: invalid response structure", flush=True)

            return None

        except Exception as e:
            is_retryable = False
            if hasattr(e, 'status_code'):
                if e.status_code == 429 or (500 <= e.status_code < 600):
                    is_retryable = True
            elif '429' in str(e) or 'rate limit' in str(e).lower():
                is_retryable = True

            if not is_retryable or attempt == max_retries - 1:
                print(f"LLM error (final): {type(e).__name__}: {e}", flush=True)
                return None
            delay = min(base_delay * (2 ** attempt), max_delay)
            jitter = random.uniform(0, 0.1 * delay)
            wait_time = delay + jitter
            print(f"LLM error (retryable): {e}. Retrying in {wait_time:.2f}s... (attempt {attempt+1}/{max_retries})", flush=True)
            time.sleep(wait_time)

    return None

def compute_reward(generated, target, source_text, topic, use_llm=True):
    global last_llm_score
    _, _, bert_f1 = bertscorer.score([generated], [target])
    bert_reward = bert_f1.item()
    if use_llm:
        new_llm = get_llm_score(generated, source_text, topic)
        if new_llm is not None:
            last_llm_score = new_llm
        llm_reward = last_llm_score
    else:
        llm_reward = 0.0
    return bert_reward + llm_reward, bert_reward, llm_reward

In [ ]:
test_df = pd.read_csv('test_data.csv')
test_dataset = Dataset.from_pandas(test_df)

Загрузка тестового датасета

In [ ]:
def build_prompt_h3(example, tokenizer):
    """Формирует промт (query) как в исходном build_prompt, но без ответа"""
    instruction = f"""You are an expert in interview analysis. Your task is to highlight the thematic codes in the interview text and corresponding quotes.

Example of a topic:
ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫБОРЫ И УСТОЙЧИВОСТЬ РОССИЙСКОЙ МОЛОДЕЖИ

Format of one piece output (strictly follow it, make several pieces):
**Общий код <general code number>: <general code name>**
"<quote text>" - **<specific code name> (Конкретный код)**

Example of the output (do not pay attention on the example text, you should follow the format):
**Общий код 1: Поколенческие характеристики, ценности и жизненные ориентиры**
"Мне кажется, большинство особо не стремятся там вот срочно, прямо сейчас там жениться, замуж, там детей и так далее. Вот. То есть как-то больше сосредоточены даже не на карьере, а на себе, на том, чтобы сложить все для себя вот так, как хочется, да. То есть не просто чтобы там выйти замуж, а чтобы выйти замуж вот по любви, чтобы все было идеально. Вот на какой-то такой идеальности что ли." - **Фокус на самореализации и качественных отношениях (конкретный код)**
"У нашего поколения все-таки все по-другому. Нам не дадут квартиру просто так. Нам не обязательно так просто получить место там где-то на работе и так далее, да. Но при этом у нас гораздо больше возможностей в плане, как сказать, чему-то научиться новому, куда-то поехать, что-то посмотреть, составить свое мнение, там высказать свое мнение даже." - **Осознание свободы выбора и новых возможностей (конкретный код)**
"У детей нынешних у них как будто меньше табу что ли. [...] они спокойно со мной могла поговорить на какие-то откровенные темы, которые мне в ее возрасте, я тоже задумывалась об этом, у меня тоже было какое-то мнение, но я боялась об этом говорить со взрослыми, потому что это было табуировано. [...] Поэтому у них мне кажется растет какое-то более свободное поколение что ли." - **Сравнение с младшим поколением: свобода от табу (конкретный код)**
"Мне кажется, что поколение у нас достаточно трудолюбивое при этом как бы. То есть если человек чего-то хочет добиться в карьерной сфере, ну, человек действительно может приложить там все усилия и добиться этого. Вот. Потому что возможностей супер много сейчас." - **Трудолюбие и вера в возможности (конкретный код)**
**Общий код 2: Жизненные выборы (образование, карьера, переезд)**
"Я хотела поступать в какой-нибудь другой город... Но мама сразу резко сказала: "Нет, на это у нас нет денег, я не смогу тебя содержать где-то там". [...] Поэтому пришлось все-таки остановиться на Питере." - **Выбор между амбициями и семейными обстоятельствами (конкретный код)**
"В [первый институт] я не закончила... мне был 21 год, оно было как раз к месту, вовремя, и университет гораздо, как сказать, дружелюбно настроенный к студентам нежели в [первый институт] это было. [...] Я сейчас в принципе университетом очень довольна." - **Осознанный выбор вуза и среды после неудачного опыта (конкретный код)**
"Сейчас для меня это очень ценно, потому что, ну, в связи с тем, почему я взяла академ, как бы у меня есть сложности иногда с выходом из дома. [...] Мне комфортно если работать с людьми, то один на один. Либо вообще работать не с людьми, а что-то другое делать. Поэтому я вот сейчас когда я снимаю, мне вообще вполне себе прекрасно." - **Выбор фриланса и фотографии из-за личных обстоятельств (конкретный код)**
"Мысли уехать были, но парень у меня все равно как-то прям ни в какую не хотел. То есть он такой очень ярый патриот своей страны. Он никуда не хочет уезжать. Вот. А я как-то наоборот, мне всегда было интересно пожить где-то в другом месте." - **Конфликт ценностей в паре: эмиграция vs патриотизм (конкретный код)**
**Общий код 3: Устойчивость и стратегии преодоления трудностей**
"Когда бабушки не стало, я поступила в университет, я параллельно там работала, я параллельно там, не знаю, танцами занималась, еще там чем-то занималась. То есть у меня все расписание было просто минута в минуту [...] И это позволяло мне просто, ну, не думать, не переживать о том, что вот бабушки не стало." - **Стратегия преодоления горя через сверхзанятость (конкретный код)**
"Мне нужно было время просто наедине с собой, чтобы меня никто не трогал и ничего не заботило. Вот. Ну, то есть, да, наверное, побыть один на один с собой." - **Необходимость уединения как способ справиться с трудностями (конкретный код)**
"У меня всегда, всегда, что бы ни произошло, даже мелочь какая-то, мне сразу надо кому-то позвонить и рассказать. То есть либо там подружке, либо маме, но вот это выговориться — это обязательно, это всегда помогает." - **Выговориться как ключевая копинг-стратегия (конкретный код)**
"Если бы не антидепрессанты, че бы я делала. [...] Плюс, наверное, очень хорошо, что у меня сейчас была возможность, то есть дать себе, собственно говоря, посидеть дома. То есть не работать, не учиться какое-то время. Вот. Это вот большое спасибо моему парню за это." - **Принятие помощи (медицинской и от партнера) как ресурс (конкретный код)**
**Общий код 4: Влияние макрособытий (СВО, корона) на идентичность и жизнь**
"Когда началась [специальная военная операция], задело меня очень сильно. Потому что, как я уже упоминала, у меня корни с Украины. [...] У меня в детстве бабушка со мной на украинском разговаривала. [...] И как-то мне бабушка прививала любовь к этой культуре, именно что вот всегда говорила: "Ты украинка". [...] И тут когда начинается [СВО] между как бы странами... это тяжело ударяет." - **Внутриличностный конфликт из-за двойной идентичности (конкретный код)**
"Очень многие уехали. Однозначно. [...] Есть немножечко ощущение, что общество как бы разделилось на тех, кто за, и тех, кто против. [...] Многим людям, в том числе и мне, которые до этого были такие аполитичные... пришлось так или иначе хоть какое-то мнение о политике составить." - **Раскол в обществе и вынужденная политизация молодежи (конкретный код)**
"В какой-то степени свободы слова лишились. То есть как бы я не против нашей власти, но я против [СВО]. Но я не могу пойти открыто об этом сказать, потому что меня сразу в кутузку посадят. Это тоже страшно. Даже если не посадят в кутузку, банально я студент вуза, меня могут отчислить." - **Страх репрессий и самоцензура (конкретный код)**
"Если корона. Очень сложно я ее восприняла морально, потому что у меня прям все, все было четко распланировано в общем. В плане тех же путешествий. [...] И все эти планы рухнули, потому что ровно как раз в 20-х числах марта началась корона... Сейчас мне, наоборот, очень комфортно сидеть дома, чтобы меня лишний раз никто не трогал." - **Крушение планов и смена паттерна поведения (от экстраверсии к интроверсии) (конкретный код)**
**Общий код 5: Представления о будущем и надежда**
"Я очень надеюсь, что [СВО] закончится. На самом деле... не знаю, может, так плохо говорить, надеюсь, не в нашу пользу. Потому что очень жалко Украину. [...] С другой стороны, как бы тоже совсем, знаешь, жить в побежденной стране тоже не очень хочется, потому что очень, очень много чего плохого это за собой повлечет для нас." - **Амбивалентность в отношении исхода войны (конкретный код)**
"Очень хочется, чтобы все-таки была возможность путешествовать заграницу. Потому что у меня ощущение, что я просто, как это, как без воздуха, сидя в одной стране. Мне прямо надо разговаривать с кем-то на английском. Вот. Знакомиться с иностранцами." - **Тоска по открытому миру и межкультурному общению (конкретный код)**
"Я очень надеюсь, что у нас с молодым человеком все вопросы будут разрешены. [...] Закончить все-таки наконец-то университет. [...] Мы хотим как-то здесь уже обустроиться, свое жилье. Вот. Там через пару лет может кого-то родить. Но главное для начала завести собаку." - **Типичные жизненные планы в условиях нестабильности (конкретный код)**
"Ну, а если не появится [возможность путешествовать], будем устраивать свою жизнь максимально хорошо здесь, как оно есть. Как-то так." - **Адаптивный оптимизм и принятие реальности (конкретный код)**

Now you should do the markup for the interview according to the plan. Important: The answer should contain only codes and quotes, without unnecessary words and repetitions.
Give the answer in Russian, as in an example.
"""

    user_content = f"""Тема интервью:
{example['topic']}

Текст интервью:
{example['transcript']}

Пожалуйста, выполни разметку интервью в указанном формате."""

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt

In [ ]:
def prepare_dataset(hf_dataset, tokenizer):
    queries, targets = [], []
    for ex in hf_dataset:
        queries.append(build_prompt_h3(ex, tokenizer))
        targets.append(ex['coding'])
    return Dataset.from_dict({"query": queries, "coding": targets})

def extract_source_and_topic(input_text):
    """
    Извлекает транскрипт и тему из поля input_text.
    Формат: "Тема интервью: ...\nТекст интервью: ..." (транскрипт может быть многострочным)
    """
    topic = ""
    transcript = ""

    lines = input_text.split('\n')
    for line in lines:
        if line.startswith("Тема интервью:"):
            topic = line.replace("Тема интервью:", "").strip()
            break

    if not topic and "Тема интервью:" in input_text:
        first_part = input_text.split("Текст интервью:")[0]
        if "Тема интервью:" in first_part:
            topic = first_part.split("Тема интервью:", 1)[1].strip()

    marker = "Текст интервью:"
    if marker in input_text:
        transcript = input_text.split(marker, 1)[1].strip()
    else:
        found = False
        transcript_lines = []
        for line in lines:
            if line.startswith("Текст интервью:"):
                found = True
                transcript_lines.append(line.replace("Текст интервью:", "").strip())
            elif found:
                transcript_lines.append(line)
        transcript = '\n'.join(transcript_lines).strip()

    return transcript, topic

Подготовка тестового датасета

In [ ]:
test_dataset = prepare_dataset(test_dataset, tokenizer)
test_source_texts = []
test_topics = []
for ex in test_dataset:
    src, top = extract_source_and_topic(ex["query"])
    test_source_texts.append(src)
    test_topics.append(top)
test_dataset = test_dataset.add_column("transcript", test_source_texts)
test_dataset = test_dataset.add_column("topic", test_topics)

Функции генерации

In [ ]:
def clear_cuda_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()

def generate_code(prompt, max_new_tokens=512, num_beams=3):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1800).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=num_beams,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated

Инференс и сбор метрик

In [ ]:
device = next(model.parameters()).device
predictions = []
all_rewards_total = []
all_rewards_bert = []
all_rewards_llm = []

for idx in tqdm(range(len(test_dataset)), desc="Evaluating"):
    batch_item = test_dataset[idx]
    query = batch_item["query"]
    target = batch_item["coding"]
    src = batch_item["transcript"]
    topic = batch_item["topic"]

    pred = generate_code(query)

    total_reward, bert_reward, llm_reward = compute_reward(pred, target, src, topic, use_llm=USE_LLM_REWARD)

    print(f'BERTScore: {bert_reward}; LLM score: {llm_reward}')

    predictions.append({
        "query": query,
        "prediction": pred,
        "target": target,
        "source_text": src,
        "topic": topic,
        "reward_total": total_reward,
        "reward_bert": bert_reward,
        "reward_llm": llm_reward
    })

pred_df = pd.DataFrame(predictions)

print("\n=== Test Metrics ===")

print(f'\nGeneration strategy: Beam Search')
print(f"Average Total Reward: {np.mean(pred_df['reward_total']):.4f}")
print(f"Average BERTScore: {np.mean(pred_df['reward_bert']):.4f}")
print(f"Average LLM Score: {np.mean(pred_df['reward_llm']):.4f}")


Evaluating:   4%|▍         | 1/23 [05:20<1:57:23, 320.17s/it]

BERTScore: 0.6267809867858887; LLM score: 1.0



Evaluating:   9%|▊         | 2/23 [12:02<2:09:03, 368.73s/it]

BERTScore: 0.6215479969978333; LLM score: 1.0



Evaluating:  13%|█▎        | 3/23 [17:39<1:58:01, 354.08s/it]

...

BERTScore: 0.6436281800270081; LLM score: 0.16666666666666666



Evaluating:  96%|█████████▌| 22/23 [2:03:17<05:24, 324.85s/it]

BERTScore: 0.6474048495292664; LLM score: 0.0



Evaluating: 100%|██████████| 23/23 [2:08:39<00:00, 335.61s/it]

BERTScore: 0.6282140016555786; LLM score: 0.0

=== Test Metrics ===

Generation strategy: Beam Search
Average Total Reward: 0.9097
Average BERTScore: 0.6278
Average LLM Score: 0.2819


In [ ]:
pred_df.to_csv('/content/drive/MyDrive/psad_project_final/Evaluation/qwen7b_prompt_engineering/7b_base_h3_predictions_with_scores.csv', index=False)
print("Результаты сохранены в 7b_base_h2_predictions_with_scores.csv")
pred_df.head()

Результаты сохранены в 7b_base_h2_predictions_with_scores.csv


,query,prediction,target,source_text,topic,reward_total,reward_bert,reward_llm
0,<|im_start|>system\nYou are an expert in inter...,"ела поступить на физика в [первый институт], н...",**Общий код 1: Переход к самостоятельности: ма...,"Интервьюер: Хорошо. Вот, согласны ли вы на зап...","Тема: МНОГООБРАЗИЕ ФОРМ, ЭФФЕКТОВ И БАРЬЕРОВ И...",1.626781,0.626781,1.0
1,<|im_start|>system\nYou are an expert in inter...,"ела поступить на физика в [первый институт], н...","**Общий код 1: Поколенческие характеристики, ц...",Интервьюер: Как всегда начнем со знакомства. Р...,"Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...",1.621548,0.621548,1.0
2,<|im_start|>system\nYou are an expert in inter...,"ела поступить на физика в [первый институт], н...",**Общий код 1: Представления о работе и критер...,"Интервьюер: Начинаю. Первый вопрос расскажи, п...",Тема: ПОКОЛЕНИЕ Z В ПОИСКАХ БАЛАНСА: УСЛОВИЯ Т...,0.628628,0.628628,0.0
3,<|im_start|>system\nYou are an expert in inter...,"ела поступить на физика в [первый институт], н...",**Общий код 1: Переход к самостоятельности: пе...,"Интервьюер: Согласие на запись даешь, верно?\n...","Тема: МНОГООБРАЗИЕ ФОРМ, ЭФФЕКТОВ И БАРЬЕРОВ И...",1.643186,0.643186,1.0
4,<|im_start|>system\nYou are an expert in inter...,"ела поступить на физика в [первый институт], н...",**Общий код 1: Переход к самостоятельности: вз...,"Интервьюер: Согласие ты даешь, верно?\nИнформа...","Тема: МНОГООБРАЗИЕ ФОРМ, ЭФФЕКТОВ И БАРЬЕРОВ И...",0.631591,0.631591,0.0
